# Working with Climate Data

## Introduction

This tutorial shows, in detail, how to make a PyEarthTools pipeline for loading climate data. It is a precursor step to creating an ML model for bias correction. It goes into a lot of detail in order to document how to process new data sources and what steps are involved. The end result is a re-usable data loading pipeline, which can then be re-used for many projects, based on the steps and assumptions shown here.

In general terms, an "analysis" refers to estimates of current conditions, "reanalysis" refers to estimates of historical conditions, "nowcasting" refers to predictions of conditions which are expected in the next 90 minutes, "short term" refers to around 6-48 hours lead time and "weather" refers to 6 hours to 10 days lead time. A variety of terms like "multi-week", "sub-seasonal" and "seasonal" refer to the period from weeks to months. Lead times longer than a few months may be referred to as "seasonal" and merge into the climate time scales. Time scales for climate modelling are typically decades into the future or longer. These terms are not accepted universally, and some people may refer to any of these time scales as climate modelling. Within this tutorial, climate data is meant to describe predictions at least a year into the future.

A "reforecast" is done to generate "what the forecast would have been". It is subtly different to a "reanalysis", because it includes the lead time component as well as the estimate at a point in time.

Climate data uses non-Gregorian calendars, which also involve the use of date/time libraries which may be unfamiliar to many users. 

This tutorial will demonstrate the loading of climate data which includes both reforecasting outputs and predictions of the future, and merging it with reanalysis data for the purposes of validation of the reforecast component of the dataset.

This will involve loading a climate prediction run from CMIP5 and ERA5 reanalysis data.


In [1]:
import pyearthtools.data as petdata
import pyearthtools.pipeline as petpipe
import site_archive_nci

In [17]:
%%capture
# This builds an accessor that can be indexed by time, filtered according to the specified parameters
# Multiple institutions, scenarios and models are unsupported but the intention is to support that in future
# Models should generally be included in a pipeline of operations rather than used directly, but we will
# explore some of the functionality of this object regardless
cmip5_model1 = petdata.archive.CMIP5(institutions='BCC', scenarios=['rcp60'], models=['bcc-csm1-1'], interval='mon', variables='tas')

In [15]:
# With an exact time, you need to pick a time actually in the dataset, for fuzzy selection see the next cell

# Under-specifying the datetime will request all source data which matches Jan 2010
# In this case, the data is monthy, with a pseudo-day-of-month of the 16th of the month
# Note for later - longitude is indexed from 0 to 360
ds_cmip_2010 = cmip5_model1['2010-01']  # Query data along primary dimension
ds_cmip_2010

In [16]:
ds_cmip_2010_2 = cmip5_model1['2010-01']  # Query data along primary dimension
ds_cmip_2010_2

<xarray.Dataset> Size: 37kB
Dimensions:    (time: 1, bnds: 2, latitude: 64, longitude: 128)
Coordinates:
  * time       (time) object 8B 2010-01-16 12:00:00
  * latitude   (latitude) float64 512B -87.86 -85.1 -82.31 ... 82.31 85.1 87.86
  * longitude  (longitude) float64 1kB 0.0 2.812 5.625 ... 351.6 354.4 357.2
    height     float64 8B 2.0
Dimensions without coordinates: bnds
Data variables:
    time_bnds  (time, bnds) object 16B dask.array<chunksize=(1, 2), meta=np.ndarray>
    lat_bnds   (time, latitude, bnds) float64 1kB dask.array<chunksize=(1, 64, 2), meta=np.ndarray>
    lon_bnds   (time, longitude, bnds) float64 2kB dask.array<chunksize=(1, 128, 2), meta=np.ndarray>
    tas        (time, latitude, longitude) float32 33kB dask.array<chunksize=(1, 64, 128), meta=np.ndarray>
Attributes: (12/24)
    institution:            Beijing Climate Center(BCC),China Meteorological ...
    institute_id:           BCC
    experiment_id:          rcp60
    source:                 bcc-csm1-1:atmosphere:  BCC_AGCM2.1 (T42L26); lan...
    model_id:               bcc-csm1-1
    forcing:                Nat Ant GHG SD Oz Sl SS Ds BC OC
    ...                     ...
    table_id:               Table Amon (11 April 2011) 1cfdc7322cf2f4a3261482...
    title:                  bcc-csm1-1 model output prepared for CMIP5 RCP6
    parent_experiment:      historical
    modeling_realm:         atmos
    realization:            1
    cmor_version:           2.5.6